In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from ultralytics import YOLO

from dataset.dataset import anonymize_dataset
from plotting import *

## Dataset

In [ ]:
def split_into_fold(data: str, n: int) -> list:
    df = pd.read_csv(data)

    sgkf = StratifiedGroupKFold(
        n_splits=n,
        shuffle=True,
        random_state=42
    )
    
    folds = []

    for train_idx, val_idx in sgkf.split(
            X=df["SOURCE"],
            y=df["SIDE"],
            groups=df["PATIENT"]):
                
        train = df.iloc[train_idx]
        val = df.iloc[val_idx]
        
        folds.append({"train": train, "valid": val})
        
    return folds

def _create_fold_class(dict_ds: dict, dir: Path, class_name: str):
    
    output = dir / class_name
    df = dict_ds[class_name]
    
    left_path = output / "left"
    right_path = output / "right"
    
    left_path.mkdir(parents=True, exist_ok=True)
    right_path.mkdir(parents=True, exist_ok=True)
        
    left_df: pd.DataFrame = df.loc[df["SIDE"] == "L"]
    right_df: pd.DataFrame = df.loc[df["SIDE"] == "R"]
    
    for _, row in left_df.iterrows():
        
        source = row["SOURCE"]
        name = row["NAME"]
        
        source = source.strip('"')
        name = name.strip()

        shutil.copy(source, left_path / name)
    
    for _, row in right_df.iterrows():
        
        source = row["SOURCE"]
        name = row["NAME"]
        
        source = source.strip('"')
        name = name.strip()

        shutil.copy(source, right_path / name)
        
def create_folds(lst_split: list, output_folder: str):
        
    output_path = Path(output_folder)
    
    for n_fold, dict_fold in enumerate(lst_split):
        
        fold_path = output_path / f"fold_{n_fold}"
                        
        # create for train
        _create_fold_class(dict_fold, fold_path, "train")
        
        # create for valid
        _create_fold_class(dict_fold, fold_path, "valid")

In [ ]:
anonymize_dataset("dataset/source/csv/sources_kfold_train.csv")
list_folds = split_into_fold("dataset/source/csv/sources_kfold_train.csv", 5)
create_folds(list_folds, "dataset/kfold")

## Training


In [ ]:
os.environ["YOLO_VERBOSE"] = "False"

def kfold_cross_validation(n: int, dataset: str, config: dict):
    dataset = Path(dataset)
    histories = []
    metrics = []
        
    for i in range(n):
        
        model = YOLO(config["model"])
        
        fold_path = dataset / f"fold_{i}"
        model_path = f"fold_{i}"
        
        model.train(
            data=fold_path,
            project="train/",
            name=model_path,
            exist_ok=True,
            epochs=config["epochs"],
            imgsz=config["imgsz"],
            batch=config["batch"],
            device=0, # GPU
            optimizer=config["optimizer"],
            verbose=False,
            fliplr=0.0
        )
        
        path = Path("runs", "classify", "train") / model_path
        df = pd.read_csv(path / "results.csv")
        df.columns = df.columns.str.strip()
        histories.append(df)
        
        metric = model.val(name=f"fold_{i}")
        
        # get accuracy recal and F1 scores
        cm = metric.confusion_matrix.matrix.T.astype(int)
        n = len(model.names)
        names = [model.names[i] for i in range(n)]
        
        idx = np.arange(len(names))
        y_true = np.repeat(idx, cm.sum(axis=1))
        y_pred = np.concatenate([np.repeat(idx, row) for row in cm]) 
        
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
        metrics.append((metric.top1, p, r, f1))
                        
    return histories, metrics

In [ ]:
config = {"model": "models/yolo11x-cls.pt",
            "optimizer": 'SGD',
            "batch": 32,
            "epochs": 25,
            "imgsz": 512}

histories, metrics = kfold_cross_validation(5, "dataset/kfold", config)
metrics

### Benchmark the best config

In [ ]:
configs = {"model": ["models/yolo11x-cls.pt"], # models/yolo26s-cls.pt", "models/yolo26m-cls.pt", "models/yolo26x-cls.pt
          "optimizer": ['SGD', 'AdamW'], 
          "batch": [8, 16, 32],
          "epochs": [100],
          "imgsz": 512}

def find_best_config(report: str):
    for model in configs["model"]:
        for optimizer in configs["optimizer"]:
            for batch in configs["batch"]:
                for epochs in configs["epochs"]:
                    
                    config = {"model": model, 
                              "optimizer": optimizer,
                              "batch": batch,
                              "epochs": epochs,
                              "imgsz": 512}
                    
                    scores, metrics = kfold_cross_validation(5, "dataset/kfold", config)
                    scores = np.asarray(scores)
                    
                    
                    fig = plot_kfold_report(histories, metrics[0], metrics, ["scores", "accuracy", "rappel", "F1 score"],
                                            show=False)
                    
                    add_text_to_report(f"model: {model}, optimizer: {optimizer}, batch: {batch}, epochs: {epochs}", path=report)
                    add_fig_to_report(fig, path=report)                  

find_best_config("report_2.html");

In [ ]:
fig = plot_kfold_report(histories, metrics[:,1], metrics[:, :1], ["accuracy", "precision", "rappel", "F1 score"])
add_text_to_report("model: yolo11x-cls.pt")
add_fig_to_report(fig)